# Experiment 8 — Asynchronous Advantage Actor-Critic (A3C)

A cleaned, standalone A3C implementation for `CartPole-v1`.

**Focus:** shared policy/value network, multiple workers, n-step returns, advantage estimation, and asynchronous updates.

In [ ]:
# %pip install torch gymnasium -q

import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.multiprocessing as mp
import gymnasium as gym

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
GAMMA = 0.99
N_STEPS = 5
LEARNING_RATE = 1e-3
WORKERS = 4
MAX_EPISODES = 200

class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(state_dim, 128), nn.ReLU())
        self.actor = nn.Linear(128, action_dim)
        self.critic = nn.Linear(128, 1)

    def forward(self, state):
        hidden = self.body(state)
        return self.actor(hidden), self.critic(hidden)

    def act(self, state):
        state_tensor = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        logits, value = self.forward(state_tensor)
        distribution = torch.distributions.Categorical(logits=logits)
        action = distribution.sample()
        return action.item(), distribution.log_prob(action), value

In [ ]:
def worker(worker_id, shared_model, optimizer, episode_counter, result_queue):
    env = gym.make('CartPole-v1')
    local_model = ActorCritic(env.observation_space.shape[0], env.action_space.n)
    local_model.load_state_dict(shared_model.state_dict())

    while True:
        with episode_counter.get_lock():
            if episode_counter.value >= MAX_EPISODES:
                break

        state, _ = env.reset(seed=SEED + worker_id)
        done = False
        episode_reward = 0.0
        log_probs, values, rewards = [], [], []

        while not done:
            action, log_prob, value = local_model.act(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            log_probs.append(log_prob)
            values.append(value)
            rewards.append(reward)
            episode_reward += reward
            state = next_state

            if len(rewards) == N_STEPS or done:
                with torch.no_grad():
                    bootstrap = 0.0 if done else local_model.act(state)[2].item()

                returns = []
                running = bootstrap
                for reward_value in reversed(rewards):
                    running = reward_value + GAMMA * running
                    returns.insert(0, running)
                returns = torch.tensor(returns, dtype=torch.float32)
                values_tensor = torch.cat(values).squeeze(-1)
                advantages = returns - values_tensor

                actor_loss = -(torch.stack(log_probs) * advantages.detach()).mean()
                critic_loss = 0.5 * advantages.pow(2).mean()
                loss = actor_loss + critic_loss

                optimizer.zero_grad()
                loss.backward()
                for local_param, shared_param in zip(local_model.parameters(), shared_model.parameters()):
                    shared_param._grad = local_param.grad
                optimizer.step()
                local_model.load_state_dict(shared_model.state_dict())
                log_probs, values, rewards = [], [], []

        with episode_counter.get_lock():
            episode_counter.value += 1
            episode = episode_counter.value
        result_queue.put((worker_id, episode, episode_reward))

    env.close()

In [ ]:
def train():
    base_env = gym.make('CartPole-v1')
    shared_model = ActorCritic(base_env.observation_space.shape[0], base_env.action_space.n)
    base_env.close()
    shared_model.share_memory()

    optimizer = torch.optim.Adam(shared_model.parameters(), lr=LEARNING_RATE)
    episode_counter = mp.Value('i', 0)
    result_queue = mp.Queue()

    processes = [mp.Process(target=worker, args=(i, shared_model, optimizer, episode_counter, result_queue)) for i in range(WORKERS)]
    for process in processes:
        process.start()

    results = []
    for _ in range(MAX_EPISODES):
        worker_id, episode, reward = result_queue.get()
        results.append(reward)
        if episode % 10 == 0:
            print(f'Episode {episode:03d} | worker={worker_id} | reward={reward:.1f}')

    for process in processes:
        process.join()

    recent = results[-10:]
    print(f'\nA3C completed successfully.')
    print(f'Last 10 episode mean reward: {sum(recent) / len(recent):.2f}')
    return results

if __name__ == '__main__':
    try:
        mp.set_start_method('spawn', force=True)
    except RuntimeError:
        pass
    rewards = train()

## Result
The notebook reports episode rewards during training and confirms completion with the mean reward from the last 10 episodes. Because CartPole is stochastic, exact rewards vary between runs.